# Nanoparticle latent depth (z) recovery

In [ ]:
import sys, os
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import json
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from core.train_utils import create_model, make_data_generator

print('TensorFlow:', tf.__version__)

## Load config and checkpoint

In [ ]:
EXP_NAME = 'NANOPARTICLE-EXP-0'
EPOCH    = None

exp_dir    = os.path.join(REPO_ROOT, 'experiments', EXP_NAME)
epochs_dir = os.path.join(exp_dir, 'epochs')
specs_path = os.path.join(exp_dir, 'specs.json')

with open(specs_path) as f:
    exp_specs = json.load(f)

h5_files = sorted([f for f in os.listdir(epochs_dir) if f.endswith('.h5')],
                  key=lambda s: int(s[2:-3]))
print(f'Available checkpoints: {h5_files}')

if EPOCH is None:
    weights_path = os.path.join(epochs_dir, h5_files[-1])
else:
    weights_path = os.path.join(epochs_dir, f'ep{EPOCH}.h5')
print(f'Loading weights from: {weights_path}')

In [ ]:
seed = exp_specs.get('seed', 0)
np.random.seed(seed)
tf.random.set_seed(seed)

model = create_model(**exp_specs['model_params'])

dg_tmp = make_data_generator(seed=seed, **exp_specs['data_generator_params'])
sample = tf.constant(dg_tmp.sample_batch_of_data(), dtype=tf.float32)
_ = model(sample, training=False)

model.load_weights(weights_path)
print('Weights loaded.')

## Load crops

In [ ]:
crops_path = os.path.join(REPO_ROOT, exp_specs['data_generator_params']['features'][0]['data_path'])
crops = np.load(crops_path).astype(np.float32)

n_components = exp_specs['data_generator_params']['n_components']
crop_size = int(round(np.sqrt(n_components)))
print(f'Crops: {crops.shape}, crop_size={crop_size}')

## Recover per-crop coordinate on the learned orbit

In [ ]:
def recover_orbit_coordinate(model, x, batch=2000):
    coords = []
    positions = None
    for i in range(0, x.shape[0], batch):
        xb = tf.constant(x[i:i+batch], dtype=tf.float32)
        lifted, _ = model(xb, training=False, analyze=True)
        resp = tf.reduce_sum(tf.abs(lifted), axis=-1).numpy()  # (b, positions)
        if positions is None:
            positions = np.arange(resp.shape[1], dtype=np.float32)
        w = resp / (resp.sum(axis=1, keepdims=True) + 1e-8)
        coords.append((w * positions[None, :]).sum(axis=1))
    return np.concatenate(coords, axis=0)

z_hat = recover_orbit_coordinate(model, crops)
print(f'Recovered z_hat: shape {z_hat.shape}, range [{z_hat.min():.2f}, {z_hat.max():.2f}]')

## Defocus proxy from raw crops

In [ ]:
def defocus_radius(crops_flat, crop_size, eps=1e-8):
    imgs = crops_flat.reshape(-1, crop_size, crop_size)
    imgs = np.clip(imgs, 0, None)
    c = crop_size // 2
    ax = np.arange(crop_size) - c
    X, Y = np.meshgrid(ax, ax)
    r2 = (X**2 + Y**2)[None, :, :]
    mass = imgs.sum(axis=(1, 2)) + eps
    return np.sqrt((imgs * r2).sum(axis=(1, 2)) / mass)

r_defocus = defocus_radius(crops, crop_size)
brightness = crops.reshape(crops.shape[0], -1).sum(axis=1)

cc = np.corrcoef(z_hat, r_defocus)[0, 1]
cc_b = np.corrcoef(z_hat, brightness)[0, 1]
print(f'corr(z_hat, defocus_radius) = {cc:.3f}')
print(f'corr(z_hat, brightness)     = {cc_b:.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(r_defocus, z_hat, s=3, alpha=0.3)
axes[0].set_xlabel('Defocus radius (2nd moment of crop)')
axes[0].set_ylabel(r'Recovered $\hat{z}$')
axes[0].set_title(f'corr = {cc:.3f}')
axes[1].scatter(brightness, z_hat, s=3, alpha=0.3)
axes[1].set_xlabel('Crop brightness (sum of intensities)')
axes[1].set_ylabel(r'Recovered $\hat{z}$')
axes[1].set_title(f'corr = {cc_b:.3f}')
plt.tight_layout(); plt.show()

## Learned 1D generator and $S_{0.75}$

In [ ]:
log_G = model.lifting_layer.log_generators.numpy()
phases = model.lifting_layer.generator_phases.numpy()

def finite_difference_generator(D):
    Delta = np.zeros((D, D))
    for i in range(D):
        Delta[i, (i + 1) % D] =  0.5
        Delta[i, (i - 1) % D] = -0.5
    return Delta

def compute_s_beta(L_learned, L_truth, beta=0.75):
    D = L_learned.shape[0]
    k_max = int(np.floor(beta * D))
    F = np.fft.fft(np.eye(D), axis=0) / np.sqrt(D)
    F_beta = F[:, :k_max]
    def project(M):
        return F_beta.conj().T @ M @ F_beta
    P_hat, P_truth = project(L_learned), project(L_truth)
    num = np.real(np.trace(P_hat.conj().T @ P_truth))
    denom = np.linalg.norm(P_hat, 'fro') * np.linalg.norm(P_truth, 'fro') + 1e-12
    return num / denom

D = log_G[0].shape[0]
s = compute_s_beta(log_G[0], finite_difference_generator(D), beta=0.75)
print(f'S_0.75 = {s:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
im0 = axes[0].imshow(log_G[0], cmap='RdBu_r')
axes[0].set_title(r'Learned $\hat{L}$ (log-generator)')
plt.colorbar(im0, ax=axes[0])
axes[1].plot(np.sort(phases[0]))
axes[1].set_title('Generator eigenvalue phases')
axes[1].set_xlabel('Eigenvalue index'); axes[1].set_ylabel('Phase (rad)')
axes[1].grid(True)
plt.tight_layout(); plt.show()